# 11 · Privacy inputs and serving tables (All countries)
**Country series · All countries** · *generated from `notebooks/country_template`: edit the template*

Differential-privacy inputs and the contracted serving tables for All countries (pipeline series, notebook 11).
The country question for privacy: a country-level release has **smaller cells** than a bank-wide one, so the same
noise costs more accuracy.

In [1]:
import sys
import time

sys.path.insert(0, "../../src")
import pandas as pd
from IPython.display import Markdown, display
from itables import show

from latam_eda import country, theme

COUNTRY = "ALL"
DATASET = "main"
PREFIX = "country"
CTRY = country.SCOPES[COUNTRY]
theme.register()
t0 = time.time()
pl = country.session(COUNTRY, DATASET)
country.prepare(pl, "features_graph_knowledge")
PRIV = ["privacy_input_complaints_country_month", "privacy_input_tx_segment_month"]
SERVING = sorted(n for n in pl.catalog()["node"] if n.startswith("serving_"))
built = pl.build_set(PRIV + SERVING)

  table     privacy_input_complaints_country_month                65,121 rows     0.2s
  table     privacy_input_tx_segment_month                     2,626,563 rows     0.7s
  table     serving_account_inquiry                              400,000 rows     0.3s
  table     serving_card_support                                 140,040 rows     0.2s
  table     serving_credit_eligibility                           150,000 rows     0.1s
  table     serving_customer_360                                 150,000 rows     0.3s
  table     serving_dispute_case                                  26,351 rows     0.1s
  table     serving_online_fraud_state                           134,515 rows     1.9s
  table     serving_recent_transactions                        4,280,701 rows     1.5s


## 1 · Cell sizes and the noise they can carry
Complaint counts per (month, category) for this country, and the relative error of a Laplace release with
sensitivity 3 (the contribution bound) at three privacy budgets.

In [2]:
cells = pl.q("""select year_month, category, sum(bounded_contribution) as complaints
                from {privacy_input_complaints_country_month} group by all""")
rows = []
for eps in (0.1, 0.5, 1.0):
    b = 3 / eps
    rows.append(
        {
            "epsilon": eps,
            "noise scale": b,
            "median cell": cells.complaints.median(),
            "median relative error %": round(100 * b / cells.complaints.median(), 1),
            "cells below 10 × scale %": round(100 * (cells.complaints < 10 * b).mean(), 1),
        }
    )
show(pd.DataFrame(rows), paging=False)
display(
    Markdown(
        f"**{CTRY.title}: {len(cells)} complaint cells, median {cells.complaints.median():.0f} complaints.** At ε = 0.5 the "
        f"median cell is released within about {100 * 6 / cells.complaints.median():.0f} %; the smaller the country, the "
        "coarser its releases must be (quarterly instead of monthly, or fewer categories) for the same budget."
    )
)

**The whole bank: 180 complaint cells, median 372 complaints.** At ε = 0.5 the median cell is released within about 2 %; the smaller the country, the coarser its releases must be (quarterly instead of monthly, or fewer categories) for the same budget.

## 2 · Serving tables

In [3]:
srv = [
    {
        "serving table": s,
        "rows": pl.rows(s),
        "contract enforced": bool(
            (pl.nodes[pl.key(s)]["config"].get("contract") or {}).get("enforced")
        ),
    }
    for s in SERVING
]
show(pd.DataFrame(srv), paging=False)
pl.ensure_until("privacy_serving")
t = pl.run_tests("privacy_serving")
display(
    Markdown(
        f"Privacy and serving tests: {', '.join(f'{k} {v}' for k, v in t.status.value_counts().items())}. "
        "A country deployment publishes only its own rows to the serving store in its residency region."
    )
)

Privacy and serving tests: pass 7. A country deployment publishes only its own rows to the serving store in its residency region.

## Findings for All countries and what to do
The computed statements above; privacy budgets are set per release and per country, and small countries need
coarser cells.

In [4]:
pl.close()
print(f"notebook time {time.time() - t0:.0f}s")

notebook time 6s
